In [ ]:
# MobileNet - Augmeted (currently)

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score,
    accuracy_score
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

SPLIT_DIR = '/content/drive/MyDrive/summer_2025_research/Dataset/dataset_split'
SAVE_DIR = '/content/drive/MyDrive/summer_2025_research/bestmodel/MobileNetV2'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)
LOSS_FUNCTION = 'categorical_crossentropy'

COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

plt.style.use('default')

def load_split_data(split_dir):
    train_df = pd.read_csv(os.path.join(split_dir, 'train_split.csv'))
    val_df = pd.read_csv(os.path.join(split_dir, 'val_split.csv'))
    test_df = pd.read_csv(os.path.join(split_dir, 'test_split.csv'))

    print(f"Train set: {len(train_df)} slices")
    print(train_df['label'].value_counts().to_string())

    print(f"Val set: {len(val_df)} slices")
    print(val_df['label'].value_counts().to_string())

    print(f"Test set: {len(test_df)} slices")
    print(test_df['label'].value_counts().to_string())

    return train_df, val_df, test_df

def build_model():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=LOSS_FUNCTION,
        metrics=['accuracy']
    )

    print(f"Model parameters: {model.count_params():,}")

    return model

# comment out for training without augmentation.
def create_data_generators(train_df, val_df, test_df):
    train_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_preprocess,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode='nearest'
    )

    val_test_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_preprocess
    )

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=True
    )

    val_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    print(f"Train batches per epoch: {len(train_generator)}")
    print(f"Val batches per epoch: {len(val_generator)}")
    print(f"Test batches: {len(test_generator)}")

    return train_generator, val_generator, test_generator

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(history.history['accuracy'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Accuracy')
    axes[0].set_title('Model Accuracy (MobileNetV2 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Loss')
    axes[1].set_title('Model Loss (MobileNetV2 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'MOBILENETV2_AUGMENTED_AL.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix (MobileNetV2 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'MOBILENETV2_AUGMENTED_CM.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_roc_curves(y_true, y_pred_proba):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}

    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"],
             label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})',
             color='navy', linestyle=':', linewidth=4)

    colors_cycle = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors_cycle):
        plt.plot(fpr[i], tpr[i], color=color, lw=2.5,
                 label=f'ROC of {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Multi-Class ROC Curves (MobileNetV2 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'MOBILENETV2_AUGMENTED_ROC.png'), dpi=300, bbox_inches='tight')
    plt.show()

def main():
    print("MobileNetV2 Training Pipeline")
    print(f"Input size: {IMG_SIZE}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Max epochs: {EPOCHS}")
    print(f"Classes: {CLASSES}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    train_df, val_df, test_df = load_split_data(SPLIT_DIR)

    train_gen, val_gen, test_gen = create_data_generators(train_df, val_df, test_df)

    model = build_model()

    print("Starting training with early stopping (patience=7)...")

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stopping],
        verbose=1
    )

    print("Test Set Evaluation")

    predictions_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_gen.classes

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")

    print("Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    plot_training_history(history)
    plot_confusion_matrix(y_true, y_pred)
    plot_roc_curves(y_true, predictions_prob)

    model_path = os.path.join(SAVE_DIR, 'MobileNetV2_Augmented.keras')
    model.save(model_path)
    print(f"Model saved to: {model_path}")

    summary_path = os.path.join(SAVE_DIR, 'MobileNetV2_training_summary.txt')
    with open(summary_path, 'w') as f:
        f.write("MobileNetV2 Training Summary\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")
        f.write(f"Balanced Accuracy: {balanced_acc:.4f}\n")
        f.write(f"Epochs trained: {len(history.history['loss'])}\n")
        f.write("Classification Report:\n")
        f.write(classification_report(y_true, y_pred, target_names=CLASSES))

    print(f"Summary saved to: {summary_path}")

    print("Training Complete")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Epochs: {len(history.history['loss'])}")
    print(f"Model: {model_path}")
    print(f"Training curves: MOBILENETV2_AUGMENTED_AL.png")
    print(f"Confusion Matrix: MOBILENETV2_AUGMENTED_CM.png")
    print(f"ROC Curves: MOBILENETV2_AUGMENTED_ROC.png")
    print(f"Summary: {summary_path}")

if __name__ == "__main__":
    main()

In [ ]:
# EfficientNetV2B0 - Augmeted (currently)

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score,
    accuracy_score
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

SPLIT_DIR = '/content/drive/MyDrive/summer_2025_research/Dataset/dataset_split'
SAVE_DIR = '/content/drive/MyDrive/summer_2025_research/bestmodel/EfficientNetV2B0'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)
LOSS_FUNCTION = 'categorical_crossentropy'

COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

plt.style.use('default')

def load_split_data(split_dir):
    train_df = pd.read_csv(os.path.join(split_dir, 'train_split.csv'))
    val_df = pd.read_csv(os.path.join(split_dir, 'val_split.csv'))
    test_df = pd.read_csv(os.path.join(split_dir, 'test_split.csv'))

    print(f"Train set: {len(train_df)} slices")
    print(train_df['label'].value_counts().to_string())

    print(f"Val set: {len(val_df)} slices")
    print(val_df['label'].value_counts().to_string())

    print(f"Test set: {len(test_df)} slices")
    print(test_df['label'].value_counts().to_string())

    return train_df, val_df, test_df

def build_model():
    base_model = EfficientNetV2B0(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=LOSS_FUNCTION,
        metrics=['accuracy']
    )

    print(f"Model parameters: {model.count_params():,}")

    return model

# comment out for training without augmentation.
def create_data_generators(train_df, val_df, test_df):
    train_datagen = ImageDataGenerator(
        preprocessing_function=efficientnet_preprocess,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode='nearest'
    )

    val_test_datagen = ImageDataGenerator(preprocessing_function=efficientnet_preprocess)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=True
    )

    val_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    print(f"Train batches per epoch: {len(train_generator)}")
    print(f"Val batches per epoch: {len(val_generator)}")
    print(f"Test batches: {len(test_generator)}")

    return train_generator, val_generator, test_generator

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(history.history['accuracy'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Accuracy')
    axes[0].set_title('Model Accuracy (EfficientNetV2B0 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Loss')
    axes[1].set_title('Model Loss (EfficientNetV2B0 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'EFFICIENTNET_AUGMENTED_AL.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix (EfficientNetV2B0 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'EFFICIENTNET_AUGMENTED_CM.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_roc_curves(y_true, y_pred_proba):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}

    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"],
             label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})',
             color='navy', linestyle=':', linewidth=4)

    colors_cycle = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors_cycle):
        plt.plot(fpr[i], tpr[i], color=color, lw=2.5,
                 label=f'ROC of {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Multi-Class ROC Curves (EfficientNetV2B0 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'EFFICIENTNET_AUGMENTED_ROC.png'), dpi=300, bbox_inches='tight')
    plt.show()

def main():
    print("EfficientNetV2B0 Training Pipeline")
    print(f"Input size: {IMG_SIZE}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Max epochs: {EPOCHS}")
    print(f"Classes: {CLASSES}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    train_df, val_df, test_df = load_split_data(SPLIT_DIR)

    train_gen, val_gen, test_gen = create_data_generators(train_df, val_df, test_df)

    model = build_model()

    print("Starting training with early stopping (patience=7)...")

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stopping],
        verbose=1
    )

    print("Test Set Evaluation")

    predictions_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_gen.classes

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")

    print("Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    plot_training_history(history)
    plot_confusion_matrix(y_true, y_pred)
    plot_roc_curves(y_true, predictions_prob)

    model_path = os.path.join(SAVE_DIR, 'EfficientNetV2B0_Augmented.keras')
    model.save(model_path)
    print(f"Model saved to: {model_path}")

    summary_path = os.path.join(SAVE_DIR, 'EfficientNetV2B0_training_summary.txt')
    with open(summary_path, 'w') as f:
        f.write("EfficientNetV2B0 Training Summary\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")
        f.write(f"Balanced Accuracy: {balanced_acc:.4f}\n")
        f.write(f"Epochs trained: {len(history.history['loss'])}\n")
        f.write("Classification Report:\n")
        f.write(classification_report(y_true, y_pred, target_names=CLASSES))

    print(f"Summary saved to: {summary_path}")

    print("Training Complete")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Epochs: {len(history.history['loss'])}")
    print(f"Model: {model_path}")
    print(f"Training curves: EFFICIENTNET_AUGMENTED_AL.png")
    print(f"Confusion Matrix: EFFICIENTNET_AUGMENTED_CM.png")
    print(f"ROC Curves: EFFICIENTNET_AUGMENTED_ROC.png")
    print(f"Summary: {summary_path}")

if __name__ == "__main__":
    main()

In [ ]:
# DenseNet121 - Augmented

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score,
    accuracy_score
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

SPLIT_DIR = '/content/drive/MyDrive/summer_2025_research/Dataset/dataset_split'
SAVE_DIR = '/content/drive/MyDrive/summer_2025_research/bestmodel/DenseNet'

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 100
CLASSES = ['cn', 'emci', 'lmci']
NUM_CLASSES = len(CLASSES)
LOSS_FUNCTION = 'categorical_crossentropy'

COLORS = {
    'purple': '#8B5CF6',
    'green': '#10B981',
    'yellow': '#F59E0B',
    'gradient': ['#8B5CF6', '#10B981', '#F59E0B']
}

plt.style.use('default')

def load_split_data(split_dir):
    train_df = pd.read_csv(os.path.join(split_dir, 'train_split.csv'))
    val_df = pd.read_csv(os.path.join(split_dir, 'val_split.csv'))
    test_df = pd.read_csv(os.path.join(split_dir, 'test_split.csv'))

    print(f"Train set: {len(train_df)} slices")
    print(train_df['label'].value_counts().to_string())

    print(f"Val set: {len(val_df)} slices")
    print(val_df['label'].value_counts().to_string())

    print(f"Test set: {len(test_df)} slices")
    print(test_df['label'].value_counts().to_string())

    return train_df, val_df, test_df

def build_model():
    base_model = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=LOSS_FUNCTION,
        metrics=['accuracy']
    )

    print(f"Model parameters: {model.count_params():,}")

    return model

def create_data_generators(train_df, val_df, test_df):
    train_datagen = ImageDataGenerator(
        preprocessing_function=densenet_preprocess,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        fill_mode='nearest'
    )

    val_test_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=True
    )

    val_generator = val_test_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    test_generator = val_test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASSES,
        shuffle=False
    )

    print(f"Train batches per epoch: {len(train_generator)}")
    print(f"Val batches per epoch: {len(val_generator)}")
    print(f"Test batches: {len(test_generator)}")

    return train_generator, val_generator, test_generator

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(history.history['accuracy'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Accuracy')
    axes[0].set_title('Model Accuracy (DenseNet121 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], color=COLORS['purple'],
                 linewidth=2.5, label='Train Loss')
    axes[1].plot(history.history['val_loss'], color=COLORS['green'],
                 linestyle='--', linewidth=2.5, label='Validation Loss')
    axes[1].set_title('Model Loss (DenseNet121 - Augmented)',
                      fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'DENSENET_AUGMENTED_AL.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix (DenseNet121 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'DENSENET_AUGMENTED_CM.png'), dpi=300, bbox_inches='tight')
    plt.show()

def plot_roc_curves(y_true, y_pred_proba):
    y_true_binarized = label_binarize(y_true, classes=range(NUM_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}

    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= NUM_CLASSES
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(all_fpr, mean_tpr)

    plt.figure(figsize=(10, 8))
    plt.plot(fpr["macro"], tpr["macro"],
             label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})',
             color='navy', linestyle=':', linewidth=4)

    colors_cycle = cycle(COLORS['gradient'])
    for i, color in zip(range(NUM_CLASSES), colors_cycle):
        plt.plot(fpr[i], tpr[i], color=color, lw=2.5,
                 label=f'ROC of {CLASSES[i]} (AUC = {roc_auc[i]:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Multi-Class ROC Curves (DenseNet121 - Augmented)',
              fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'DENSENET_AUGMENTED_ROC.png'), dpi=300, bbox_inches='tight')
    plt.show()

def main():
    print("DenseNet121 Training Pipeline")
    print(f"Input size: {IMG_SIZE}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Max epochs: {EPOCHS}")
    print(f"Classes: {CLASSES}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    train_df, val_df, test_df = load_split_data(SPLIT_DIR)

    train_gen, val_gen, test_gen = create_data_generators(train_df, val_df, test_df)

    model = build_model()

    print("Starting training with early stopping (patience=7)...")

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stopping],
        verbose=1
    )

    print("Test Set Evaluation")

    predictions_prob = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(predictions_prob, axis=1)
    y_true = test_gen.classes

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")

    print("Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    plot_training_history(history)
    plot_confusion_matrix(y_true, y_pred)
    plot_roc_curves(y_true, predictions_prob)

    model_path = os.path.join(SAVE_DIR, 'DenseNet121_Augmented.keras')
    model.save(model_path)
    print(f"Model saved to: {model_path}")

    summary_path = os.path.join(SAVE_DIR, 'DenseNet121_training_summary.txt')
    with open(summary_path, 'w') as f:
        f.write("DenseNet121 Training Summary\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")
        f.write(f"Balanced Accuracy: {balanced_acc:.4f}\n")
        f.write(f"Epochs trained: {len(history.history['loss'])}\n")
        f.write("Classification Report:\n")
        f.write(classification_report(y_true, y_pred, target_names=CLASSES))

    print(f"Summary saved to: {summary_path}")

    print("Training Complete")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Epochs: {len(history.history['loss'])}")
    print(f"Model: {model_path}")
    print(f"Training curves: DENSENET_AUGMENTED_AL.png")
    print(f"Confusion Matrix: DENSENET_AUGMENTED_CM.png")
    print(f"ROC Curves: DENSENET_AUGMENTED_ROC.png")
    print(f"Summary: {summary_path}")

if __name__ == "__main__":
    main()